In [ ]:

# IMPORTS


# Standard Library


import os
import re
import json
import time
import sqlite3
import hashlib
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict


# Type Hinting


from typing import Any, Dict, List, Tuple, Optional

# Numerical Computing


import numpy as np

# Data Processing


import pandas as pd

# Natural Language Processing


import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Sentence Embeddings


from sentence_transformers import SentenceTransformer

# Vector Database (Semantic Search)


import faiss


# Knowledge Graph


import networkx as nx


# NVIDIA API Client


from openai import OpenAI


# Progress Bars

from tqdm import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

# Project

PROJECT_NAME = "Personal Memory AI"
PROJECT_VERSION = "1.0.0"

# Directories

NOTES_FOLDER = "notes"
DATABASE_FILE = "personal_memory.db"

# NVIDIA API

NVIDIA_API_KEY = "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"

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

LLM_MODEL = "meta/llama-3.3-70b-instruct"

LLM_TEMPERATURE = 0.2
LLM_MAX_TOKENS = 4096

# Embedding Model

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

EMBEDDING_DIMENSION = 384

# spaCy

SPACY_MODEL = "en_core_web_sm"

# TF-IDF

MAX_FEATURES = 1000

NGRAM_RANGE = (1, 2)

STOP_WORDS = "english"

# Importance Scoring

SUMMARY_LENGTH = 3

MIN_NOTE_LENGTH = 30

# Semantic Search

TOP_K_RESULTS = 10

SIMILARITY_THRESHOLD = 0.60

# Memory Consolidation

CONSOLIDATE_EVERY = 25

# Random Seed

RANDOM_SEED = 42

# DATABASE

def initialize_database():

    # Connect to SQLite database

    connection = sqlite3.connect(DATABASE_FILE)

    cursor = connection.cursor()

    # Enable foreign keys

    cursor.execute("PRAGMA foreign_keys = ON")

    # Create Notes table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS notes (

        note_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_hash TEXT UNIQUE,

        note_date TEXT,

        original_text TEXT,

        clean_text TEXT,

        summary TEXT,

        importance REAL,

        created_at TEXT

    )
    """)

    # Create Memories table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS memories (

        memory_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_id INTEGER,

        memory_type TEXT,

        memory_value TEXT,

        confidence REAL,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Topics table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS topics (

        topic_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_id INTEGER,

        topic TEXT,

        score REAL,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Entities table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS entities (

        entity_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_id INTEGER,

        entity TEXT,

        entity_type TEXT,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Relationships table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS relationships (

        relationship_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_id INTEGER,

        source TEXT,

        relation TEXT,

        target TEXT,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Embeddings table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS embeddings (

        note_id INTEGER PRIMARY KEY,

        embedding BLOB,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Observations table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS observations (

        observation_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_id INTEGER,

        observation TEXT,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Predictions table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS predictions (

        prediction_id INTEGER PRIMARY KEY AUTOINCREMENT,

        prediction TEXT,

        confidence REAL,

        created_at TEXT

    )
    """)

    # Create Indexes

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_notes_date ON notes(note_date)")

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_memories_type ON memories(memory_type)")

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_topics_topic ON topics(topic)")

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_entities_name ON entities(entity)")

    # Save changes

    connection.commit()

    print("Database initialized successfully.")

    return connection



# MEMORY SYSTEM INITIALIZATION


def initialize_memory_system():

    global nlp

    global embedding_model

    global vector_index

    global knowledge_graph

    global tfidf_vectorizer

    nlp = spacy.load(SPACY_MODEL)

    embedding_model = SentenceTransformer(EMBEDDING_MODEL)

    vector_index = faiss.IndexFlatL2(EMBEDDING_DIMENSION)

    knowledge_graph = nx.MultiDiGraph()

    tfidf_vectorizer = TfidfVectorizer(

        max_features=MAX_FEATURES,

        ngram_range=NGRAM_RANGE,

        stop_words=STOP_WORDS

    )

    print("Memory system initialized.")

# ============================================================
# CALL LLM
# ============================================================

# LLM CLIENT

client = OpenAI(

    base_url=NVIDIA_BASE_URL,

    api_key=NVIDIA_API_KEY

)

def call_llm(

    system_prompt,

    user_prompt,

    temperature = LLM_TEMPERATURE,

    max_tokens = LLM_MAX_TOKENS

):

    try:

        response = client.chat.completions.create(

            model = LLM_MODEL,

            temperature = temperature,

            max_tokens = max_tokens,

            messages = [

                {

                    "role": "system",

                    "content": system_prompt

                },

                {

                    "role": "user",

                    "content": user_prompt

                }

            ]

        )

        return response.choices[0].message.content.strip()

    except Exception as error:

        print(f"LLM Error : {error}")

        return None
    
# ============================================================
# NOTES PROCESSING
# ============================================================

def import_notes():

    notes = []

    NOTES_FOLDER = "notes"

    notes_path = Path(NOTES_FOLDER)

    if not notes_path.exists():

        raise FileNotFoundError(f"{NOTES_FOLDER} folder not found.")

    note_files = sorted(notes_path.glob("notes_*.txt"))

    print(f"Found {len(note_files)} notes.")

    for file in tqdm(note_files):

        text = file.read_text(

            encoding = "utf-8"

        ).strip()

        notes.append(

            {

                "filename": file.name,

                "text": text

            }

        )

    return notes


    